# 11장 웹부록 | 문장의 감정 점수 매기기

「영화로 보는 AI 리터러시」 11장 '마음을 가진 기계?: 감성 AI와 로봇 윤리'의 코랩 실습입니다.

**쓰는 법**

- 코드를 몰라도 괜찮습니다. 위에서부터 차례로 셀 왼쪽의 ▶(실행) 버튼을 누르세요.
- `# ← 이 숫자를 바꿔 보세요` 표시가 있는 곳만 고치고, 그 셀부터 다시 실행해 보세요.
- 처음부터 다시 하고 싶으면 메뉴에서 **런타임 → 모두 실행**을 누르세요.
- 따로 설치할 것은 없습니다. 그래프 글자는 영어로 나오고, 설명은 이 글에 우리말로 적었습니다.

**이 실습에서 할 일**

1. 책 3-3절의 옛 방식, **단어 점수** 방식으로 우리말 문장의 감정을 점수로 바꿉니다.
2. '최고였어', '최고로 별로였어', '참 잘했네…' 같은 문장에서 어디서 틀리는지 찾습니다.
3. 예문에서 문장 전체의 말투를 배우는 **언어 모델 방식**과 비교합니다.
4. 내가 쓴 문장으로 두 방식을 속여 보는 '감정 퀴즈'를 합니다.

> **알아 둘 점**: 책에서 말한 대규모 언어 모델은 크기가 수 GB가 넘고, 인터넷에서 내려받아야 합니다. 이 실습은 설치와 인터넷 없이 누구나 1분 안에 돌릴 수 있도록, **예문 120개로 이 자리에서 바로 학습시키는 아주 작은 언어 모델**을 씁니다. 진짜 대규모 언어 모델보다 훨씬 작고 서툴지만, '단어 하나'가 아니라 '글자 조각의 흐름'을 보고 판단한다는 점은 같습니다.

## 1단계. 단어 점수 사전 만들기

처음의 감정 분석은 단어마다 점수를 정해 두고 더하는 방식이었습니다. 책에 나온 대로 '최고'는 +2점, '별로'는 −1점입니다. 문장에 그 단어가 들어 있으면 점수를 더합니다. 합이 0보다 크면 긍정, 작으면 부정, 0이면 '모르겠음'입니다.

👉 **바꿔 볼 곳**: `WORD_SCORES`에 단어를 더하거나 점수를 바꿔 보세요. 단어 뒤에 붙는 말(-였어, -네 등)은 신경 쓰지 않아도 됩니다. '좋'만 적어도 '좋았어', '좋은'에 모두 걸립니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

WORD_SCORES = {   # ← 단어와 점수를 바꿔 보세요
    "최고": +2, "감동": +2, "훌륭": +2, "행복": +2, "사랑": +2,
    "좋": +1, "재미있": +1, "재밌": +1, "잘": +1, "고마": +1, "추천": +1, "기쁘": +1,
    "별로": -1, "나쁘": -1, "싫": -1, "지루": -1, "아깝": -1, "슬프": -1,
    "최악": -2, "실망": -2, "짜증": -2, "화나": -2, "재미없": -2,
}

def word_score(sentence):
    """문장 속 단어마다 사전의 점수를 찾아 더합니다."""
    total, hits = 0, []
    for word in sentence.replace("…", " ").replace(".", " ").split():
        for key, s in WORD_SCORES.items():
            if key in word:
                total += s; hits.append(f"{key}({s:+d})")
    return total, hits

def word_label(total):
    return "긍정" if total > 0 else ("부정" if total < 0 else "모르겠음")

for s in ["최고였어", "최고로 별로였어", "참 잘했네…"]:
    total, hits = word_score(s)
    print(f"{s:<12} → 점수 {total:+d} ({word_label(total)})   찾은 단어: {', '.join(hits) or '없음'}")

## 2단계. 열 문장으로 시험하기

사람이 정답을 붙여 둔 열 문장으로 시험해 봅니다. '정답'은 사람이 읽었을 때의 뜻입니다. '참 잘했네…'는 칭찬처럼 보이지만, 말줄임표와 말투로 보아 비꼬는 말이므로 '부정'으로 적었습니다.

In [ ]:
TEST = [("최고였어", "긍정"), ("최고로 별로였어", "부정"), ("참 잘했네…", "부정"), ("나쁘지 않았어", "긍정"),
        ("시간 가는 줄 몰랐다", "긍정"), ("돈이 아깝지 않은 영화", "긍정"), ("정말 재미있었고 감동적이었어", "긍정"),
        ("두 시간 내내 지루했다", "부정"), ("기대를 안 했는데 의외로 좋았어", "긍정"), ("좋은 배우들로 이런 영화를 만들다니", "부정")]

rows = []
for i, (s, answer) in enumerate(TEST, 1):
    total, hits = word_score(s)
    rows.append({"번호": f"S{i}", "문장": s, "정답": answer, "단어 점수": total,
                 "단어 방식 판단": word_label(total), "맞음?": "O" if word_label(total) == answer else "X"})
word_table = pd.DataFrame(rows)
display(word_table)
print(f"단어 점수 방식: 10문장 중 {(word_table['맞음?'] == 'O').sum()}개 맞힘")

**읽는 법**: '최고로 별로였어'는 +2와 −1이 더해져 긍정이 됩니다. '나쁘지 않았어'는 '나쁘'만 보고 부정이 됩니다. 단어 점수 방식은 **단어를 하나씩 따로** 보기 때문에, '않다'처럼 뜻을 뒤집는 말이나 비꼬는 말투를 읽지 못합니다.

## 3단계. 예문에서 말투를 배우는 작은 언어 모델 만들기

이번에는 3장의 **지도학습**으로 모델을 만듭니다. 사람이 '긍정', '부정'을 붙여 둔 예문 120개를 보여 주고 스스로 규칙을 찾게 합니다.

- 모델은 문장을 2~3글자짜리 **글자 조각**으로 잘라 봅니다. '나쁘지 않'은 '나쁘', '쁘지', '지 않' 같은 조각이 됩니다.
- 그래서 '나쁘'만이 아니라 '지 않'이 함께 나오는 흐름, '…'으로 끝나는 말투 같은 것을 점수에 넣을 수 있습니다.
- 사람이 단어 점수를 정해 주지 않습니다. 모델이 예문에서 조각마다 점수(가중치)를 스스로 정합니다.

아래 셀에서 예문을 직접 볼 수 있습니다. 2단계의 시험 문장은 예문에 넣지 않았습니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

POSITIVE_EXAMPLES = [
    '정말 재미있게 봤어요',
    '배우들 연기가 훌륭했다',
    '다시 보고 싶은 영화',
    '친구에게 꼭 추천하고 싶어요',
    '음악이 너무 좋았어',
    '마지막 장면에서 눈물이 났다 감동이야',
    '오늘 수업 정말 유익했어요',
    '기대 이상이었어',
    '웃다가 배가 아플 뻔했다',
    '이 집 음식 맛있다',
    '덕분에 하루가 행복했어',
    '생각보다 괜찮았어',
    '나쁘지 않네',
    '나쁘지 않은 선택이었다',
    '싫지 않았어',
    '후회하지 않을 영화',
    '지루하지 않았어요',
    '별로일 줄 알았는데 좋았어',
    '기대 안 했는데 대박이었다',
    '걱정했는데 의외로 재밌었어',
    '처음엔 지루했지만 끝은 멋졌다',
    '고마워 정말 큰 도움이 됐어',
    '설명이 친절해서 이해가 잘 됐어요',
    '완전 마음에 들어',
    '가격 대비 훌륭합니다',
    '두 번 봐도 재밌다',
    '가족과 함께 보기 좋은 영화',
    '최고의 하루였어',
    '배우 얼굴만 봐도 즐겁다',
    '선물 받고 너무 기뻤어',
    '이번 발표 아주 잘했어 축하해',
    '시원하고 깔끔한 맛',
    '기다린 보람이 있었어',
    '울고 웃었다 강력 추천',
    '마음이 따뜻해지는 이야기',
    '이런 영화 또 나왔으면 좋겠다',
    '감독님 사랑합니다',
    '빈틈없이 잘 만든 작품',
    '결말까지 완벽했어',
    '노래가 계속 귀에 맴돌 만큼 좋다',
    '상상 이상으로 멋졌어',
    '시간이 순식간에 지나갔다',
    '몰입해서 봤어요',
    '친절한 직원 덕분에 기분 좋았다',
    '아이들이 정말 좋아했어요',
    '다음 편이 너무 기대된다',
    '흠잡을 데가 없다',
    '역시 믿고 보는 배우',
    '오랜만에 크게 웃었다',
    '만족스러운 결과야',
    '진짜 최고다',
    '훌륭한 연출에 박수를 보냅니다',
    '돈이 하나도 안 아까웠다',
    '좋다 좋아',
    '행복한 결말이라 다행이야',
    '깜짝 놀랄 만큼 좋았어요',
    '작은 극장이었지만 감동은 컸다',
    '설레는 마음으로 봤어',
    '정말 고마운 선생님',
    '재미와 감동 둘 다 잡았다',
]
NEGATIVE_EXAMPLES = [
    '정말 지루했어',
    '돈이 아깝다',
    '두 시간이 아까웠다',
    '배우 연기가 어색했어',
    '다시는 안 볼 거야',
    '최악의 영화',
    '기대했는데 실망이야',
    '줄거리가 엉망이야',
    '음식이 너무 짜고 맛없어',
    '오늘 하루 정말 짜증 났어',
    '설명이 너무 어려워서 하나도 모르겠다',
    '화가 나서 잠이 안 와',
    '별로였어',
    '최고로 지루했다',
    '최고로 실망스러운 결말',
    '최고급이라더니 엉망이었다',
    '좋은 줄 알았는데 아니었어',
    '재미있을 줄 알았는데 지루했다',
    '좋지 않았어',
    '즐겁지 않았다',
    '재미없어',
    '웃기지도 않네',
    '참 잘하는 짓이네…',
    '아주 훌륭하시네요…',
    '대단하다 대단해… 또 늦다니',
    '참 친절도 하셔라…',
    '잘도 그러겠다…',
    '고맙기도 하네… 덕분에 다 망쳤어',
    '퍽이나 재밌겠다',
    '그래 너 참 잘났다',
    '중간에 나오고 싶었다',
    '졸려서 혼났다',
    '시끄럽고 불친절했어요',
    '배송이 너무 늦어서 속상해',
    '다시는 이 가게 안 가',
    '광고만 요란했지 내용이 없다',
    '결말이 허무하다',
    '보는 내내 불편했어',
    '이걸 영화라고 만들었나',
    '음악만 좋고 나머지는 별로',
    '싼 게 비지떡',
    '제값을 못 한다',
    '연기도 각본도 엉성하다',
    '시간 낭비였어',
    '기분이 나빴다',
    '슬프고 우울한 하루',
    '피곤하고 힘들어',
    '실수투성이 발표였다',
    '친구랑 싸워서 속상해',
    '너무 비싸고 양도 적다',
    '재미있다는 사람 이해가 안 간다',
    '예고편이 전부였다',
    '끝까지 보기 힘들었다',
    '추천하고 싶지 않다',
    '좋은 재료로 이런 맛을 내다니',
    '이 정도면 사기다',
    '불친절한 서비스에 실망',
    '지루하고 뻔한 이야기',
    '눈물 나게 재미없다',
    '진짜 별로야',
]

def train_model(pos, neg):
    model = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3)),
                          LogisticRegression(C=10, max_iter=1000))
    model.fit(pos + neg, [1] * len(pos) + [0] * len(neg))
    return model

model = train_model(POSITIVE_EXAMPLES, NEGATIVE_EXAMPLES)
print(f"예문 {len(POSITIVE_EXAMPLES) + len(NEGATIVE_EXAMPLES)}개로 학습 완료!")

def model_score(sentence):
    """긍정일 확률(0~1)을 -1 ~ +1 점수로 바꿉니다. +1에 가까울수록 긍정입니다."""
    p = model.predict_proba([sentence])[0, 1]
    return 2 * p - 1

def model_label(score, unsure=0.1):
    return "긍정" if score > unsure else ("부정" if score < -unsure else "모르겠음")

for s in ["최고였어", "최고로 별로였어", "참 잘했네…"]:
    sc = model_score(s)
    print(f"{s:<12} → 모델 점수 {sc:+.2f} ({model_label(sc)})")

## 4단계. 두 방식을 나란히 비교하기

같은 열 문장을 두 방식으로 채점합니다. 그래프는 문장 번호(S1~S10)로 표시했습니다. 막대가 오른쪽(+)이면 긍정, 왼쪽(−)이면 부정으로 판단한 것입니다. 단어 점수는 크기를 맞추려고 2로 나눠 그렸습니다.

In [ ]:
word_table["모델 점수"] = [round(model_score(s), 2) for s in word_table["문장"]]
word_table["모델 판단"] = [model_label(v) for v in word_table["모델 점수"]]
word_table["모델 맞음?"] = np.where(word_table["모델 판단"] == word_table["정답"], "O", "X")
display(word_table[["번호", "문장", "정답", "단어 방식 판단", "맞음?", "모델 점수", "모델 판단", "모델 맞음?"]])
print(f"단어 점수 방식: {(word_table['맞음?'] == 'O').sum()} / 10,   작은 언어 모델: {(word_table['모델 맞음?'] == 'O').sum()} / 10")

y = np.arange(len(word_table))[::-1]
plt.figure(figsize=(7, 4.5))
plt.barh(y + 0.2, np.clip(word_table["단어 점수"] / 2, -1, 1), height=0.4, label="word scores (/2)")
plt.barh(y - 0.2, word_table["모델 점수"], height=0.4, label="small language model")
for yy, ans in zip(y, word_table["정답"]):
    plt.text(1.05, yy, "+" if ans == "긍정" else "-", va="center", fontsize=12, weight="bold")
plt.yticks(y, word_table["번호"]); plt.axvline(0, color="gray")
plt.xlim(-1.1, 1.2); plt.xlabel("negative  <--  score  -->  positive")
plt.title("Same sentences, two methods (right column = true answer)"); plt.legend(loc="lower left")
plt.show()

**읽는 법**: 작은 언어 모델은 '최고로 별로였어', '나쁘지 않았어', '참 잘했네…'처럼 단어 점수 방식이 틀린 문장을 여럿 바로잡습니다. 하지만 이 모델도 틀립니다. 어떤 문장을 틀렸는지 표에서 찾아보세요.

- '시간 가는 줄 몰랐다'에는 감정 단어가 하나도 없습니다. 예문에 비슷한 말투가 없으면 모델도 짐작하기 어렵습니다.
- 예문에 '최고로 지루했다'처럼 '최고'가 부정 문장에 쓰인 경우가 섞여 있어서, 모델이 '최고'를 헷갈리기도 합니다. 모델은 **보여 준 예문만큼만** 압니다. 6장의 데이터 치우침과 같은 문제입니다.

## 5단계. 모델의 속 들여다보기

모델이 글자 조각마다 매긴 점수를 꺼내 봅니다. +가 크면 '이 조각이 보이면 긍정 쪽', −가 크면 '부정 쪽'이라는 뜻입니다. 사람이 정해 준 1단계의 사전과 비교해 보세요.

In [ ]:
vec, clf = model.named_steps["tfidfvectorizer"], model.named_steps["logisticregression"]
pieces = np.array(vec.get_feature_names_out())
weights = clf.coef_[0]
order = np.argsort(weights)
top = pd.DataFrame({
    "긍정 쪽 조각": [repr(p) for p in pieces[order[::-1][:12]]],
    "긍정 점수": np.round(weights[order[::-1][:12]], 2),
    "부정 쪽 조각": [repr(p) for p in pieces[order[:12]]],
    "부정 점수": np.round(weights[order[:12]], 2),
})
display(top)
print("작은따옴표 안의 빈칸은 띄어쓰기입니다. '…'(말줄임표) 조각이 어느 쪽에 있는지 찾아보세요.")

**읽는 법**: '좋았', '지루했', '실'(실망), '…' 같은 조각은 그럴듯합니다. 그런데 '어요'나 '요 '처럼 감정과 상관없는 조각도 긍정 쪽에 올라 있습니다. 예문 속 긍정 문장에 마침 '~어요' 말투가 많았기 때문입니다. 모델은 감정이 아니라 **말투라는 지름길**을 배운 셈입니다. 그렇다면 '~어요'로 끝나는 부정 문장으로 모델을 속일 수 있지 않을까요?

## 6단계. 감정 퀴즈: 내 문장으로 모델 속이기

이제 여러분 차례입니다. 사람은 쉽게 알아듣지만 기계는 헷갈릴 만한 문장을 써 보세요. 비꼬기, 이중 부정, 감정 단어 없는 감정, 반어법, 줄임말, 이모티콘을 써 볼 수 있습니다.

아래 예시 문장은 모두 **영화 후기**라고 생각하고 읽어 보세요. 영화 후기에서 "팝콘이 제일 맛있었다"는 칭찬이 아닙니다.

👉 **바꿔 볼 곳**: `MY_QUIZ`에 (문장, 정답)을 적으세요. 정답은 "긍정" 또는 "부정"입니다. 몇 줄이든 더해도 됩니다.

In [ ]:
MY_QUIZ = [   # ← 내 문장과 정답으로 바꿔 보세요
    ("와 진짜 대단하다… 또 지각이네", "부정"),
    ("별로 기대 안 했는데 별 다섯 개 줍니다", "긍정"),
    ("환불하고 싶어요", "부정"),
    ("팝콘이 제일 맛있었다", "부정"),
    ("감독 다음 작품은 안 볼래요", "부정"),
    ("ㅋㅋㅋ 꿀잼", "긍정"),
]

rows = []
for s, answer in MY_QUIZ:
    w = word_label(word_score(s)[0]); m = model_label(model_score(s))
    rows.append({"내 문장": s, "정답": answer, "단어 방식": w, "언어 모델": m,
                 "단어 방식 속음": "속았다!" if w != answer else "", "언어 모델 속음": "속았다!" if m != answer else ""})
quiz = pd.DataFrame(rows)
display(quiz)
print(f"단어 방식을 속인 문장: {(quiz['단어 방식 속음'] != '').sum()}개 / 언어 모델을 속인 문장: {(quiz['언어 모델 속음'] != '').sum()}개")

## 더 해 보기: 속은 문장을 가르쳐 주면?

모델이 속은 문장을 정답과 함께 예문에 더하고 다시 학습시키면 어떻게 될까요? 진짜 AI 회사도 틀린 사례를 모아 다시 학습시킵니다. 다만 모델이 그 문장만 외우는 것인지, 비슷한 다른 문장까지 알아듣게 되는 것인지는 따로 확인해야 합니다.

👉 **바꿔 볼 곳**: `CHECK`에 '가르쳐 준 문장과 비슷하지만 다른 문장'을 적어 보세요.

In [ ]:
extra_pos = [s for s, a in MY_QUIZ if a == "긍정" and model_label(model_score(s)) != a]
extra_neg = [s for s, a in MY_QUIZ if a == "부정" and model_label(model_score(s)) != a]
print("다시 가르칠 문장:", extra_pos + extra_neg)

model = train_model(POSITIVE_EXAMPLES + extra_pos, NEGATIVE_EXAMPLES + extra_neg)

CHECK = [("본전 생각이 나요", "부정"), ("음료수가 제일 맛있었다", "부정")]   # ← 바꿔 보세요
for s, a in MY_QUIZ + CHECK:
    sc = model_score(s)
    tag = "(가르친 문장)" if s in extra_pos + extra_neg else ("(새 문장)" if (s, a) in CHECK else "")
    print(f"{s:<26} 정답 {a} → 다시 배운 모델 {model_label(sc)} ({sc:+.2f}) {tag}")

## 생각해 볼 것

1. 단어 점수 방식은 '왜 그렇게 판단했는지'를 단어 목록으로 바로 보여 줍니다. 언어 모델은 더 잘 맞히지만 이유를 설명하기 어렵습니다. 콜센터가 화난 고객을 찾는 데 쓴다면, 둘 중 무엇을 고르겠습니까? 틀렸을 때 누가 손해를 보는지 함께 생각해 보세요.
2. 이 실습의 모델은 글만 봤습니다. 사람은 '참 잘했네…'를 들을 때 말투, 표정, 앞뒤 상황을 함께 봅니다. 책 3-3절의 **멀티모달** 감정 인식이 이 문제를 얼마나 풀 수 있을지, 그래도 남는 한계는 무엇일지 쓰세요.
3. 유럽연합은 직장과 학교에서 감정을 인식하는 AI를 원칙적으로 금지했습니다(책 4-1절). 여러분이 쓴 과제 글이나 단체 대화방의 감정을 학교가 자동으로 분석한다면 찬성하겠습니까? 감정 퀴즈에서 모델을 속인 경험을 근거로 들어 보세요.